# 05 · Entrenamiento difusión: DDPM condicional por clase, 128×128 (reanudable)

**Fase CRISP-DM:** 4. Modelado.

**Técnica:** modelo de difusión en espacio de píxeles, entrenado **desde cero**:
- **DDPM** (Ho et al., 2020): predicción del ruido ε, 1000 pasos, *schedule* coseno (`squaredcos_cap_v2` en `diffusers`).
- Red: **U-Net** `UNet2DModel` de `diffusers`, canales (64, 128, 256, 256, 512), atención en 16×16 y 8×8, ~72.7 M parámetros (misma del piloto).
- **Condicionamiento por clase** con *embedding* sumado al del paso temporal (`num_class_embeds`), más una **clase nula** usada el 10 % de los pasos → permite **classifier-free guidance** (Ho y Salimans, 2022) al generar.
- **EMA** de los pesos (decay 0.9999), práctica estándar en DDPM; los checkpoints para evaluar usan los pesos EMA.
- Generación posterior con **DDIM** (Song et al., 2021), 50 pasos.

**Entrenamiento:** AdamW lr 1e-4 con *warmup* lineal de 1000 pasos, batch 32, bf16, *gradient clipping* 1.0. Sin volteo horizontal. Solo **train**.

**Reanudable:** guarda el estado completo (modelo, EMA, optimizador, paso, generadores aleatorios) en Drive cada `SAVE_EVERY` pasos.
Si la sesión se cae, **vuelve a ejecutar todo**: continúa desde el último estado. Además guarda los pesos EMA cada `SNAP_EVERY` pasos para la selección con validación.

**Duración:** 100k pasos ≈ 17.6 h en L4 (1.58 pasos/s medidos en el piloto) ≈ 32 unidades.
**Antes de empezar:** GPU **L4**, versión **2026.07**; pestaña abierta.

## Paso 1 · Entorno

In [ ]:
import sys, os, json, shutil, hashlib, zipfile, time, math, random, io
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn.functional as F
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "⚠️ SIN GPU")

Python: 3.12.13 | PyTorch: 2.11.0+cu128 | CUDA: 12.8
GPU: NVIDIA L4


In [ ]:
%pip install -q diffusers

## Paso 2 · Parámetros

In [ ]:
SEED = 42
RES = 128
TOTAL_STEPS = int(os.environ.get("TOTAL_STEPS", 100_000))
BATCH = int(os.environ.get("BATCH", 32))
LR = 1e-4
WARMUP = int(os.environ.get("WARMUP", 1000))
EMA_DECAY = 0.9999
P_UNCOND = 0.1
SAVE_EVERY = int(os.environ.get("SAVE_EVERY", 2500))     # estado completo para reanudar (~26 min)
SNAP_EVERY = int(os.environ.get("SNAP_EVERY", 10_000))   # pesos EMA para selección con validación
PRINT_EVERY_S = 300 if IN_COLAB else 5
CHANNELS = tuple(int(x) for x in os.environ.get("CHANNELS", "64,128,256,256,512").split(","))
RUN_NAME = "ddpm_cond128_seed42"

## Paso 3 · Drive y dataset (mismo zip verificado que usa la GAN)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/tesis_retail")
    LOCAL_DIR = Path("/content/data")
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./tesis_retail")).resolve()
    LOCAL_DIR = Path(os.environ.get("LOCAL_DIR", "./data_local")).resolve()
LOGS = PROJECT_DIR / "logs"
RUN_DIR = PROJECT_DIR / "checkpoints" / RUN_NAME
(RUN_DIR / "ema").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "muestras").mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

LOG03 = json.loads((LOGS / "03_piloto.json").read_text())
ZIP = LOCAL_DIR / LOG03["datasets"]["train_zip"]
if not ZIP.exists():
    shutil.copy2(PROJECT_DIR / "datasets" / ZIP.name, ZIP)
assert sha256_file(ZIP) == LOG03["datasets"]["train_sha256"], "⚠️ El dataset no coincide con el del notebook 03"

from PIL import Image
with zipfile.ZipFile(ZIP) as zf:
    etiquetas = dict(json.loads(zf.read("dataset.json"))["labels"])
    nombres = sorted(etiquetas)
    X = np.stack([np.asarray(Image.open(io.BytesIO(zf.read(n))).convert("RGB")) for n in nombres])
    Y = np.array([etiquetas[n] for n in nombres], dtype=np.int64)
N_CLASSES = int(Y.max()) + 1
X_train = torch.from_numpy(X).permute(0, 3, 1, 2).contiguous().to(DEVICE)   # uint8
Y_train = torch.from_numpy(Y).to(DEVICE)
assert X_train.shape[1:] == (3, RES, RES)
print(f"Dataset verificado ✔ | {len(Y)} imágenes | {N_CLASSES} clases | carpeta: {RUN_DIR}")

Mounted at /content/drive
Dataset verificado ✔ | 3060 imágenes | 17 clases | carpeta: /content/drive/MyDrive/tesis_retail/checkpoints/ddpm_cond128_seed42


## Paso 4 · Modelo, EMA, optimizador y *scheduler*

In [ ]:
import copy, diffusers
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler

def nueva_unet():
    return UNet2DModel(
        sample_size=RES, in_channels=3, out_channels=3, layers_per_block=2,
        block_out_channels=CHANNELS,
        down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D"),
        num_class_embeds=N_CLASSES + 1)               # índice N_CLASSES = clase nula (CFG)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
unet = nueva_unet().to(DEVICE)
ema = copy.deepcopy(unet).eval().requires_grad_(False)
opt = torch.optim.AdamW(unet.parameters(), lr=LR)
sched = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")
NULL = N_CLASSES
GEN = torch.Generator(device=DEVICE).manual_seed(SEED)
AMP = dict(device_type=DEVICE.type, dtype=torch.bfloat16)
N_PARAMS = sum(p.numel() for p in unet.parameters())
print("diffusers:", diffusers.__version__, "| parámetros (M):", round(N_PARAMS / 1e6, 1))

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


diffusers: 0.39.0 | parámetros (M): 72.7


## Paso 5 · ¿Empezar o reanudar?

In [ ]:
ESTADO = RUN_DIR / "estado_ultimo.pt"
paso = 0
if ESTADO.exists():
    ck = torch.load(ESTADO, map_location=DEVICE, weights_only=False)
    unet.load_state_dict(ck["unet"]); ema.load_state_dict(ck["ema"]); opt.load_state_dict(ck["opt"])
    paso = ck["paso"]
    GEN.set_state(ck["gen"].cpu())                       # los estados RNG deben volver a CPU
    torch.set_rng_state(ck["rng_cpu"].cpu())
    if DEVICE.type == "cuda" and ck.get("rng_cuda") is not None:
        torch.cuda.set_rng_state(ck["rng_cuda"].cpu())
    print(f"Se reanuda desde el paso {paso:,}")
else:
    print("No hay estado previo: entrenamiento nuevo.")
if paso >= TOTAL_STEPS:
    print(f"✔ Ya se alcanzó el tope ({paso:,} pasos).")

No hay estado previo: entrenamiento nuevo.


## Paso 6 · Funciones de entrenamiento, guardado y muestreo

In [ ]:
def lote(bs):
    idx = torch.randint(0, X_train.shape[0], (bs,), device=DEVICE, generator=GEN)
    return X_train[idx].float() / 127.5 - 1.0, Y_train[idx]

def lr_en(p):
    return LR * min(1.0, (p + 1) / WARMUP)

def paso_entrenamiento(p):
    for g in opt.param_groups:
        g["lr"] = lr_en(p)
    x, y = lote(BATCH)
    nulo = torch.rand(BATCH, device=DEVICE, generator=GEN) < P_UNCOND
    y = torch.where(nulo, torch.full_like(y, NULL), y)
    ruido = torch.randn(x.shape, device=DEVICE, generator=GEN)
    t = torch.randint(0, sched.config.num_train_timesteps, (BATCH,), device=DEVICE, generator=GEN)
    xt = sched.add_noise(x, ruido, t)
    with torch.autocast(**AMP):
        pred = unet(xt, t, class_labels=y).sample
    loss = F.mse_loss(pred.float(), ruido)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
    opt.step()
    with torch.no_grad():
        for pe, pm in zip(ema.parameters(), unet.parameters()):
            pe.lerp_(pm, 1 - EMA_DECAY)
    return loss.item()

def guardar_atomico(obj, ruta):
    tmp = ruta.with_suffix(".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, ruta)

def guardar_estado(p):
    guardar_atomico({"unet": unet.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(), "paso": p,
                     "gen": GEN.get_state(), "rng_cpu": torch.get_rng_state(),
                     "rng_cuda": torch.cuda.get_rng_state() if DEVICE.type == "cuda" else None,
                     "config": {"channels": CHANNELS, "n_classes": N_CLASSES, "res": RES}}, ESTADO)

@torch.no_grad()
def muestrear(modelo, y, pasos=50, guia=2.0, seed=0):
    ddim = DDIMScheduler.from_config(sched.config); ddim.set_timesteps(pasos)
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.randn(len(y), 3, RES, RES, device=DEVICE, generator=g)
    y_null = torch.full_like(y, NULL)
    for t in ddim.timesteps:
        with torch.autocast(**AMP):
            e = modelo(torch.cat([x, x]), t, class_labels=torch.cat([y, y_null])).sample.float()
        e_c, e_u = e.chunk(2)
        x = ddim.step(e_u + guia * (e_c - e_u), t, x).prev_sample
    return x.clamp(-1, 1)

def snapshot_ema(p):
    guardar_atomico({"ema": ema.state_dict(), "paso": p,
                     "config": {"channels": CHANNELS, "n_classes": N_CLASSES, "res": RES}},
                    RUN_DIR / "ema" / f"ema_{p:07d}.pt")
    import torchvision.utils as vutils
    y = torch.arange(N_CLASSES, device=DEVICE).repeat(4 if IN_COLAB else 1)
    x = muestrear(ema, y, pasos=50 if IN_COLAB else 2)
    vutils.save_image((x + 1) / 2, RUN_DIR / "muestras" / f"muestras_{p:07d}.png", nrow=N_CLASSES)

## Paso 7 · Entrenar
Muestra una línea de progreso cada ~5 min. Puedes detener la celda (■) cuando quieras: se guarda el estado y se reanuda después.

In [8]:
LOG_PATH = LOGS / "05_difusion_progreso.csv"
if not LOG_PATH.exists():
    LOG_PATH.write_text("fecha,paso,perdida_media,pasos_por_s\n")
unet.train()
t_ultimo, perdidas, p_ini, t_ini = time.time(), [], paso, time.time()
try:
    while paso < TOTAL_STEPS:
        perdidas.append(paso_entrenamiento(paso))
        paso += 1
        if paso % SAVE_EVERY == 0 or paso == TOTAL_STEPS:
            guardar_estado(paso)
        if paso % SNAP_EVERY == 0 or paso == TOTAL_STEPS:
            unet.eval(); snapshot_ema(paso); unet.train()
        if time.time() - t_ultimo > PRINT_EVERY_S:
            vel = (paso - p_ini) / (time.time() - t_ini)
            faltan = (TOTAL_STEPS - paso) / max(vel, 1e-9) / 3600
            pm = float(np.mean(perdidas)); perdidas = []
            print(f"[{datetime.now().strftime('%H:%M')}] paso {paso:,}/{TOTAL_STEPS:,} | pérdida {pm:.4f} | "
                  f"{vel:.2f} pasos/s | faltan ≈ {faltan:.1f} h", flush=True)
            with open(LOG_PATH, "a") as f:
                f.write(f"{datetime.now(timezone.utc).isoformat(timespec='seconds')},{paso},{pm:.5f},{vel:.3f}\n")
            t_ultimo = time.time()
except KeyboardInterrupt:
    print("Detenido manualmente.")
finally:
    if paso > p_ini:
        guardar_estado(paso)
        print(f"Estado guardado en el paso {paso:,}. Esta sesión: {(time.time()-t_ini)/3600:.2f} h")

[18:35] paso 465/100,000 | pérdida 0.3830 | 1.55 pasos/s | faltan ≈ 17.9 h
[18:40] paso 936/100,000 | pérdida 0.0859 | 1.56 pasos/s | faltan ≈ 17.7 h
[18:45] paso 1,407/100,000 | pérdida 0.0605 | 1.56 pasos/s | faltan ≈ 17.5 h
[18:50] paso 1,878/100,000 | pérdida 0.0532 | 1.56 pasos/s | faltan ≈ 17.4 h
[18:55] paso 2,349/100,000 | pérdida 0.0489 | 1.56 pasos/s | faltan ≈ 17.3 h
[19:00] paso 2,815/100,000 | pérdida 0.0467 | 1.56 pasos/s | faltan ≈ 17.3 h
[19:05] paso 3,286/100,000 | pérdida 0.0442 | 1.56 pasos/s | faltan ≈ 17.2 h
[19:10] paso 3,757/100,000 | pérdida 0.0440 | 1.56 pasos/s | faltan ≈ 17.1 h
[19:15] paso 4,228/100,000 | pérdida 0.0429 | 1.57 pasos/s | faltan ≈ 17.0 h
[19:20] paso 4,699/100,000 | pérdida 0.0418 | 1.57 pasos/s | faltan ≈ 16.9 h
[19:25] paso 5,165/100,000 | pérdida 0.0410 | 1.56 pasos/s | faltan ≈ 16.8 h
[19:30] paso 5,636/100,000 | pérdida 0.0418 | 1.56 pasos/s | faltan ≈ 16.8 h
[19:35] paso 6,107/100,000 | pérdida 0.0396 | 1.57 pasos/s | faltan ≈ 16.7 h
[19

## Paso 8 · Resumen y registro

In [9]:
snaps = sorted((RUN_DIR / "ema").glob("ema_*.pt"))
print(f"Paso actual: {paso:,} | snapshots EMA: {len(snaps)}")
for s in snaps[-5:]:
    print("  ", s.name, f"({s.stat().st_size/2**20:.0f} MB)")
mallas = sorted((RUN_DIR / "muestras").glob("*.png"))
if mallas and IN_COLAB:
    from IPython.display import Image as IPImage, display
    print("Última malla (4 muestras por clase, columnas = clases):", mallas[-1].name)
    display(IPImage(filename=str(mallas[-1]), width=1000))

registro = LOGS / "05_difusion_registro.json"
hist = json.loads(registro.read_text()) if registro.exists() else {"sesiones": []}
hist.update({"run": RUN_NAME, "tecnica": "DDPM condicional + CFG (p_uncond 0.1), U-Net diffusers, EMA 0.9999",
             "parametros_M": round(N_PARAMS / 1e6, 1), "channels": CHANNELS, "batch": BATCH, "lr": LR,
             "warmup": WARMUP, "total_steps": TOTAL_STEPS, "diffusers": diffusers.__version__})
hist["sesiones"].append({"fin_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                         "desde_paso": p_ini, "hasta_paso": paso,
                         "versiones": {"python": sys.version.split()[0], "torch": torch.__version__}})
registro.write_text(json.dumps(hist, indent=2, ensure_ascii=False))
print("Registro actualizado:", registro)

Output hidden; open in https://colab.research.google.com to view.

**Al terminar:** desconecta la GPU (*Entorno de ejecución → Desconectar y borrar*).